# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Do NOT treat as dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields. 

We'll list the available record sets (`@id`), their descriptions, and for each, the fields (`@id` and their data types) they expose. This overview is essential for referencing data by `@id` in subsequent analysis.

In [ ]:
record_sets = list(metadata.record_sets)
if not record_sets:
    print("No record sets explicitly attached to root metadata. Attempting to discover them through file objects.")
    # mlcroissant will generally attach record sets to files referenced in 'distribution' or 'hasPart'
    # Let's explore dataset.files and see if record_sets can be inferred
    files = list(metadata.files)
    for file in files:
        print(f"File: {file.name}, @id: {file.id}")
        for rs in getattr(file, 'record_sets', []):
            print(f"  Record Set: {rs.name}, @id: {rs.id}, description: {rs.description if hasattr(rs, 'description') else ''}")
            print("    Fields:")
            for field in rs.fields:
                type_ = getattr(field, 'data_type', None)
                print(f"      - {field.name} (@id: {field.id}, data_type: {type_})")
    # Collect discovered record set ids
    discovered_record_set_ids = []
    for file in files:
        for rs in getattr(file, 'record_sets', []):
            discovered_record_set_ids.append(rs.id)
else:
    for rs in record_sets:
        print(f"Record Set: {rs.name}, @id: {rs.id}, description: {rs.description if hasattr(rs, 'description') else ''}")
        print("  Fields:")
        for field in rs.fields:
            type_ = getattr(field, 'data_type', None)
            print(f"    - {field.name} (@id: {field.id}, data_type: {type_})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

If record sets were discovered in the previous section, we will use those.

In [ ]:
# Compose a list of record set ids
record_set_ids = []
record_set_names = []
files = list(metadata.files)
for file in files:
    for rs in getattr(file, 'record_sets', []):
        record_set_ids.append(rs.id)
        record_set_names.append(rs.name)

print("Available Record Sets (by @id):")
for name, rid in zip(record_set_names, record_set_ids):
    print(f"- {name} (@id: {rid})")

dataframes = {}
for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Record Set '{record_set}' loaded with shape: {df.shape}")

# For demonstration, let's choose the first available record set
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
    print(f"\nFields (columns) in the chosen record set '@id': {chosen_record_set_id}")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())
else:
    print('No record sets were found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We will:
- Select a numeric field by its `@id` (referenced from the earlier metadata overview),
- Filter rows using a threshold value,
- Normalize this field (z-score),
- Optionally group by a categorical field (if present in the data), and
- Show the resulting DataFrame.

In [ ]:
# Set your record set @id and select a numeric field by its column name (@id).
record_set_id = chosen_record_set_id
# Detect a numeric field automatically from the DataFrame if possible
df = dataframes[record_set_id]
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if numeric_cols:
    numeric_field = numeric_cols[0]  # Example: pick the first numeric column
else:
    print('No numeric fields found. Please adjust the numeric_field assignment.')
    numeric_field = df.columns[0]

print(f"Using numeric field: {numeric_field}")
threshold = df[numeric_field].quantile(0.9)  # Use top 10% as example threshold

filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a suitable group field (categorical)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
group_field = categorical_cols[0] if categorical_cols else None
if group_field:
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll use matplotlib and seaborn for quick plots. Here, we visualize the distribution of the selected numeric field, and a boxplot by group (if a categorical field is present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field distribution
plt.figure(figsize=(6,4))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the record sets in the mass spectrometry dataset using `mlcroissant`.
- We inspected available fields, loaded the primary record set, and performed basic EDA including filtering and normalization of numeric columns.
- The notebook demonstrated dynamic referencing by `@id`, making it robust against schema changes.
- You can extend this template for deeper analysis or use specific field `@id`s of interest as needed.

For more information about this dataset or `mlcroissant`, see the [Croissant documentation](https://mlcommons.org/croissant/) or the dataset's own documentation.